# 🛫 Model 2: K-Means Customer Segmentation
### KQ Water Bottling Plant (Project Kifaru) — Machine Learning Intelligence
---
**Objective:** Group clients using RFM analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
print("Libraries successfully imported!")

## 1. Load Dataset from GitHub
We extract the historical transactional logs directly from the system's database CSV export.

In [ ]:
dataset_url = "https://raw.githubusercontent.com/aawbed/kq-sales-order-intelligence/main/kq_historical_sales_2025_2026.csv"
df_orders = pd.read_csv(dataset_url)

df_orders['date'] = pd.to_datetime(df_orders['Date'])
df_orders = df_orders.rename(columns={
    'Order ID': 'order_id',
    'Customer Name': 'customer_name',
    'Account Type': 'account_type',
    'Product Name': 'product_name',
    'Quantity': 'quantity',
    'Unit Price (KSh)': 'unit_price_kes',
    'Total Amount (KSh)': 'total_amount_kes',
    'Total Litres': 'total_litres'
})

print(f"Dataset successfully loaded!\nTotal Records: {len(df_orders)}")
df_orders.head()

## 2. Model Training & Evaluation

In [ ]:
# Calculate RFM
snapshot_date = df_orders['date'].max() + timedelta(days=1)
rfm = df_orders.groupby('customer_name').agg({
    'date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'total_amount_kes': 'sum'
}).reset_index()
rfm.columns = ['Customer', 'Recency', 'Frequency', 'Monetary']

# Scale features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# Fit K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Assign readable labels based on monetary value
cluster_order = rfm.groupby('Cluster')['Monetary'].mean().sort_values(ascending=False).index
labels_map = {cluster_order[0]: 'VIP / High-Volume', cluster_order[1]: 'Consistent Regulars', cluster_order[2]: 'At-Risk / Dormant'}
rfm['Segment'] = rfm['Cluster'].map(labels_map)

# Plot
plt.figure(figsize=(12, 6))
palette = {'VIP / High-Volume': '#2b8a3e', 'Consistent Regulars': '#1c7ed6', 'At-Risk / Dormant': '#e03131'}
sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Segment', palette=palette, s=180, alpha=0.9)

for i, row in rfm.iterrows():
    plt.text(row['Frequency'] + 0.3, row['Monetary'], row['Customer'], fontsize=8.5, alpha=0.85)

plt.title("Customer Segmentation: Frequency vs. Total Spend (KES)", fontweight='bold')
plt.xlabel("Order Frequency (Total Orders)")
plt.ylabel("Total Monetary Value (KSh)")
plt.legend()
plt.show()

print("\n=== Segment Summary ===")
display(rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean().round(2))